# Fine-tune PaddleOCR's recognizer for OCRead - Kaggle version

**Before running** (right-hand panel, *Session options*):
1. **Accelerator: GPU T4 x2** or **GPU P100**.
2. **Internet: On** (needs a phone-verified Kaggle account).
3. **Add Input** → your dataset with `ocread_rec_dataset.zip` (Create → New Dataset → upload the zip made with `python -m QC_tools.make_rec_dataset`; Kaggle unpacks it).

Then *Run All*. At the end a download link for `ocread_rec_model.zip` appears (also in the *Output* panel). See docs/finetune_ocread.md for using it in the app.

In [ ]:
!nvidia-smi
import urllib.request
urllib.request.urlopen('https://github.com', timeout=10); print('internet OK')

## 1. Install PaddlePaddle (GPU) and PaddleOCR's training code
If the install fails, get the pip command for this CUDA version from https://www.paddlepaddle.org.cn/en/install/quick (pip, Linux, GPU).

In [ ]:
!pip install -q paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
import paddle; paddle.utils.run_check()

In [ ]:
%cd /kaggle/working
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd /kaggle/working/PaddleOCR
!pip install -q -r requirements.txt

## 2. Find the dataset

In [ ]:
import glob, os, zipfile
hits = glob.glob('/kaggle/input/**/train.txt', recursive=True)
if not hits:                                   # zip not unpacked by Kaggle: unpack it here
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'No dataset found - add your uploaded dataset with "Add Input"'
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/ocread_data')
    hits = glob.glob('/kaggle/working/ocread_data/**/train.txt', recursive=True)
DATA = os.path.dirname(hits[0])
print('dataset:', DATA)
!wc -l {DATA}/train.txt {DATA}/val.txt && head -3 {DATA}/train.txt

## 3. Pretrained PP-OCRv5 mobile recognizer (starting point)

In [ ]:
!mkdir -p pretrain_models && wget -q -nc -P pretrain_models https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams
!ls -la pretrain_models

## 4. Fine-tune
Batch size 32 (fits a 16 GB GPU). You can stop once `acc` stops improving and run the export cell - it exports the best checkpoint so far.
If a setting name is rejected (`KeyError`, e.g. for `sampler`), remove that setting and copy the error to Claude.

In [ ]:
CFG = 'configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml'
COMMON = (f'Global.character_dict_path={DATA}/dict.txt Global.use_space_char=True '
          'Global.max_text_length=25')
!python tools/train.py -c {CFG} -o {COMMON} \
    Global.pretrained_model=pretrain_models/PP-OCRv5_mobile_rec_pretrained.pdparams \
    Global.epoch_num=20 Global.save_model_dir=/kaggle/working/output/ocread_rec Global.eval_batch_step="[0,500]" \
    Optimizer.lr.learning_rate=0.0003 \
    Train.dataset.data_dir={DATA} Train.dataset.label_file_list="['{DATA}/train.txt']" \
    Eval.dataset.data_dir={DATA} Eval.dataset.label_file_list="['{DATA}/val.txt']" \
    Train.loader.batch_size_per_card=32 Train.sampler.first_bs=32 Eval.loader.batch_size_per_card=32

## 5. Export and download

In [ ]:
!python tools/export_model.py -c {CFG} -o {COMMON} \
    Global.pretrained_model=/kaggle/working/output/ocread_rec/best_accuracy Global.save_inference_dir=/kaggle/working/ocread_rec_model
%cd /kaggle/working
!ls -la ocread_rec_model && zip -q -r ocread_rec_model.zip ocread_rec_model
from IPython.display import FileLink
FileLink('ocread_rec_model.zip')